# 00. Список линий Xe II, Ba I, Ba II по измеренному спектру

Отождествление пиков спектра с линиями NIST, проверка бленд и классификация по
наличию атомных данных:

* **A** — верхний уровень 5p⁴6p: σ из основного и из метастабилей есть у Ванга, нужны A (ноутбук 01);
* **B** — σ только из основного (Ванг);
* **C** — верхнего уровня нет у Ванга (6d, 7s, 8s, 4f, 7p, (¹D₂)6d) — расширенная модель (ноутбук 03);
* **D** — высокие l (5g, 6g, 6f, 7g) — не считаем;
* **Ba** — линии бария (ноутбуки 04, 05).

Результат: `lines.csv` и `lines.md` (оформленный список) в папке `NIST`.

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
REPO    = Path("/data/pydbsr_calc/DBSR_src")          # клон репозитория pydbsr
CRM     = REPO / "examples" / "crm"                   # эта папка: nbtools.py, run_model.py, models.py
DATA    = Path("/data/pydbsr_calc/data")              # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
SPECTRUM, SHEET = DATA / "A25_second.xlsx", "data2"   # измеренный спектр
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST (nist_*.csv) и список линий
RUNS    = Path("/data/pydbsr_calc/runs/crm")          # результаты расчётов (большой диск)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import crm, reference
import nbtools as nb
%matplotlib inline

In [ ]:
import line_list as ll
rows = ll.build(XLSX, NIST, SPECTRUM, SHEET, Te=1.1)
ll.write_csv(rows, NIST / "lines.csv")
ll.write_md(rows, NIST / "lines.md", f"{SPECTRUM.name}, лист {SHEET}")
import collections
print(collections.Counter((r["class"], r.get("observed")) for r in rows))

## Спектр и отмеченные линии (выберите участок)

In [ ]:
w, I = ll.read_spectrum(SPECTRUM, SHEET)
LO, HI = 540, 565                                       # нм
m = (w > LO) & (w < HI)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(w[m], I[m], lw=0.6, color="k")
colors = {"A": "tab:blue", "B": "tab:cyan", "C": "tab:orange", "D": "tab:gray", "Ba": "tab:red"}
for r in rows:
    if LO < r["wl_nm"] < HI and r.get("observed") in ("да", "бленда"):
        ax.axvline(r["wl_nm"] - 0.045, color=colors[r["class"]], lw=0.6, alpha=0.7)
        if r.get("role") or r["class"] == "Ba":
            ax.text(r["wl_nm"] - 0.045, I[m].max() * 0.9, f"{r['ion']} {r['wl_nm']:.2f}", rotation=90,
                    fontsize=7, color=colors[r["class"]])
ax.set_yscale("log"); ax.set_xlabel("λ, нм (шкала прибора)"); ax.set_ylabel("отсчёты")
plt.show()

## Запрошенные линии

In [ ]:
for wl in (452.421, 659.501, 669.432, 545.045, 545.090, 561.667, 553.548, 455.403, 493.408):
    r = min(rows, key=lambda r: abs(r["wl_nm"] - wl))
    print(f"{r['ion']:6s} {r['wl_nm']:8.3f}  {r['upper']} -> {r['lower']}  класс {r['class']}  "
          f"№ {r.get('upper_no', '')}->{r.get('lower_no', '')}  S/B {r.get('snr')}  бленда: {r.get('blend', '')}")